# 03 - Model B

Evaluasi Whisper Base multilingual, lalu latih LoRA dan validasi dengan data lokal. Training penuh ditujukan untuk Colab T4; GPU lokal 4 GB mungkin memerlukan penyesuaian. Pilih checkpoint dari dev.

Audio train di atas 30 detik belum digunakan karena perlu penyelarasan potongan dan transkrip. Dev dan test tetap menggunakan audio panjang secara utuh.

## Konfigurasi awal

| Pengaturan | Nilai |
| --- | --- |
| Data | FLEURS Indonesia dan Inggris |
| Model | Whisper Base multilingual + LoRA |
| LoRA | rank 8, alpha 16, dropout 0,05; q_proj dan v_proj |
| Optimizer | AdamW, learning rate 0,0001 |
| Batch | 1, akumulasi 8 langkah |
| Persiapan fitur | 4 thread, cache lokal |
| Batas awal | 3 epoch |
| Pilihan checkpoint | Rata-rata WER dev terendah |
| GPU | T4 jika tersedia |

Konfigurasi ini titik awal untuk eksperimen. Test dijalankan setelah pemilihan menggunakan dev.


## Pengaturan

Siapkan pustaka, data lokal, dan lokasi hasil.


In [ ]:
%pip -q install 'transformers==4.46.3' 'peft==0.13.2' 'accelerate==1.1.1' 'safetensors>=0.4.5'


In [ ]:
from pathlib import Path
from collections import defaultdict
import csv, hashlib, json, math, os, random, time, unicodedata
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
import torch
from torch.utils.data import Dataset, DataLoader
from IPython.display import display, Markdown, HTML
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = root / "data"
RUN_ROOT = root / "checkpoints" / "model_b"
CACHE_ROOT = root / "data" / "preprocessed" / "model_b"
RUN_ID = "whisper_base_lora_v1"
EPOCHS = 6
BATCH_SIZE = 1
ACCUMULATION = 8
LEARNING_RATE = 0.0001
SEED = 42
MODEL_ID = "openai/whisper-base"
LANGUAGES = ("id_id", "en_us")
LANGUAGE_NAMES = {"id_id": "Indonesia", "en_us": "Inggris"}
WHISPER_LANG = {"id_id": "indonesian", "en_us": "english"}
CONFIG = {"model": MODEL_ID, "rank": 8, "alpha": 16, "dropout": 0.05,
          "targets": ["q_proj", "v_proj"], "batch": BATCH_SIZE,
          "accumulation": ACCUMULATION, "lr": LEARNING_RATE, "seed": SEED}
RUN_DIR = RUN_ROOT / RUN_ID
print("Perangkat:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("Data:", DATA_ROOT)
print("Hasil:", RUN_DIR)


## Data dan preprocessing

Baca FLEURS Indonesia dan Inggris. Audio diubah menjadi mono 16 kHz dan fitur Whisper.


In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFC", text).casefold().replace("’", "'")
    return " ".join("".join(c if c.isalnum() or c == "'" else " " for c in text).split())

duration_samples = {}


def read_split(split):
    rows = []
    for language in LANGUAGES:
        path = DATA_ROOT / language / f"{split}.tsv"
        audio_dir = DATA_ROOT / language / "audio" / split
        print(f"Memeriksa {LANGUAGE_NAMES[language]} / {split}...", flush=True)
        available = {entry.name for entry in audio_dir.glob("*.wav")}
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for number, fields in enumerate(csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE), 1):
                if len(fields) != 7 or not fields[0].isdigit():
                    raise ValueError(f"TSV tidak valid: {path}, baris {number}")
                filename = fields[1]
                if Path(filename).name != filename or not filename.lower().endswith(".wav"):
                    raise ValueError(f"Nama WAV tidak valid: {path}, baris {number}")
                audio = audio_dir / filename
                if filename not in available:
                    raise FileNotFoundError(audio)
                if not fields[5].isdigit() or not normalize_text(fields[3]):
                    raise ValueError(f"Durasi atau transkrip tidak valid: {path}, baris {number}")
                duration_samples[audio] = int(fields[5])
                rows.append((language, audio, normalize_text(fields[3]), fields[0]))
    keys = [(language, path.name) for language, path, _, _ in rows]
    if len(set(keys)) != len(keys):
        raise ValueError(f"Ada nama WAV ganda di {split}. Tinjau data.")
    return rows

def read_audio(path):
    rate, pcm = wavfile.read(path)
    samples = pcm.astype(np.float32)
    if np.issubdtype(pcm.dtype, np.integer):
        info = np.iinfo(pcm.dtype)
        if info.min == 0:
            midpoint = (info.max + 1) / 2
            samples = (samples - midpoint) / midpoint
        else:
            samples /= -info.min
    if samples.ndim == 2:
        samples = samples.mean(axis=1)
    if rate <= 0 or not samples.size or not np.isfinite(samples).all():
        raise ValueError(f"Audio tidak valid: {path}")
    if rate != 16000:
        divisor = math.gcd(rate, 16000)
        samples = resample_poly(samples, 16000 // divisor, rate // divisor).astype(np.float32)
    return samples

def feature_for(path):
    relative = path.relative_to(DATA_ROOT)
    signature = f"{path.stat().st_size}-{path.stat().st_mtime_ns}"
    destination = CACHE_ROOT / relative.parent / f"{path.stem}-{signature}.npz"
    if destination.is_file():
        with np.load(destination, allow_pickle=False) as cached:
            return cached["features"], cached["attention_mask"]
    samples = read_audio(path)
    long_audio = len(samples) > 30 * 16000
    encoded = processor.feature_extractor(
        samples, sampling_rate=16000, return_tensors="np", return_attention_mask=True,
        truncation=not long_audio, padding="longest" if long_audio else "max_length"
    )
    features, attention_mask = encoded.input_features[0], encoded.attention_mask[0]
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(".tmp")
    with temporary.open("wb") as stream:
        np.savez(stream, features=features, attention_mask=attention_mask)
    temporary.replace(destination)
    return features, attention_mask

def fingerprint(rows):
    payload = [(lang, p.name, text, sid) for lang, p, text, sid in rows]
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False).encode()).hexdigest()

def eligible_rows(rows, split):
    eligible = []
    excluded = defaultdict(int)
    for row in rows:
        if duration_samples[row[1]] <= 30 * 16000:
            eligible.append(row)
        else:
            excluded[row[0]] += 1
    for language in LANGUAGES:
        print(f"{split} {LANGUAGE_NAMES[language]}: {excluded[language]} audio > 30 detik belum digunakan.")
    return eligible

train_rows = read_split("train")
dev_rows = read_split("dev")
display(Markdown("| Data | Jumlah |\n| --- | ---: |\n"
                 f"| Train | {len(train_rows):,} |\n| Dev | {len(dev_rows):,} |"))


## Model dan metrik

Siapkan Whisper multilingual, adapter LoRA, dan pengukuran WER serta CER.


In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID)
tokenizers = {}
for language in LANGUAGES:
    tokenizer = WhisperProcessor.from_pretrained(MODEL_ID, language=WHISPER_LANG[language], task="transcribe").tokenizer
    tokenizers[language] = tokenizer

def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, expected in enumerate(reference, 1):
        current = [i]
        for j, actual in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (expected != actual)))
        previous = current
    return previous[-1]

from collections import deque
from concurrent.futures import ThreadPoolExecutor


def prefetch_features(rows, workers=4, prefetch=8):
    source = iter(rows)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = deque()
        for _ in range(min(prefetch, len(rows))):
            row = next(source)
            pending.append((row, pool.submit(feature_for, row[1])))
        while pending:
            row, future = pending.popleft()
            features = future.result()
            next_row = next(source, None)
            if next_row is not None:
                pending.append((next_row, pool.submit(feature_for, next_row[1])))
            yield row, features


def progress_batches(items, title, total=None):
    if total is None:
        total = len(items)
    done = 0
    started = time.perf_counter()
    def render(status):
        percent = done * 100 / total if total else 100
        return HTML(f"<b>{title}</b> — {status} · {done}/{total} · {percent:.0f}% · {time.perf_counter()-started:.0f} dtk"
                    f"<div style='height:8px;background:#ddd'><div style='height:100%;width:{percent:.1f}%;background:#008b83'></div></div>")
    view = display(render("berjalan"), display_id=True)
    try:
        for item in items:
            yield item
            done += 1
            if done % 10 == 0 or done == total:
                view.update(render("berjalan"))
    finally:
        view.update(render("selesai" if done == total else "terhenti"))

@torch.inference_mode()
def evaluate(model, rows, title):
    model.eval()
    counts = defaultdict(lambda: [0, 0, 0, 0])
    examples = {}
    batches = prefetch_features(rows)
    for (language, path, reference, _), (features, attention_mask) in progress_batches(
        batches, title, total=len(rows)
    ):
        features = torch.from_numpy(features).unsqueeze(0).to(DEVICE)
        attention_mask = torch.from_numpy(attention_mask).unsqueeze(0).to(DEVICE)
        ids = model.generate(
            input_features=features, attention_mask=attention_mask,
            language=WHISPER_LANG[language], task="transcribe", max_new_tokens=440,
            return_timestamps=features.shape[-1] > 3000
        )
        prediction = normalize_text(processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])
        words = reference.split()
        word_errors = edit_distance(words, prediction.split())
        counts[language][0] += word_errors
        counts[language][1] += len(words)
        counts[language][2] += edit_distance(reference, prediction)
        counts[language][3] += len(reference)
        if words:
            sample = {"reference": reference, "prediction": prediction,
                      "wer": word_errors / len(words), "filename": path.name}
            if language not in examples:
                examples[language] = {"best": sample, "worst": sample}
            if sample["wer"] < examples[language]["best"]["wer"]:
                examples[language]["best"] = sample
            if sample["wer"] > examples[language]["worst"]["wer"]:
                examples[language]["worst"] = sample
    result = {lang: {"wer": w/n, "cer": c/m, "reference_words": n}
              for lang, (w,n,c,m) in counts.items()}
    return {"per_language": result, "macro_wer": sum(result[l]["wer"] for l in LANGUAGES)/len(LANGUAGES)}, examples

def show_results(title, metrics, examples):
    lines = [f"### {title}", "", "| Bahasa | WER | CER |", "| --- | ---: | ---: |"]
    for lang in LANGUAGES:
        row = metrics["per_language"][lang]
        lines.append(f"| {LANGUAGE_NAMES[lang]} | {row['wer']:.1%} | {row['cer']:.1%} |")
    display(Markdown("\n".join(lines)))
    for lang in LANGUAGES:
        display(Markdown(f"**{LANGUAGE_NAMES[lang]}**"))
        for key, title in (("best", "Terbaik"), ("worst", "Terburuk")):
            sample = examples[lang][key]
            print(f"{title} · WER {sample['wer']:.1%} · {sample['filename']}")
            print(f"Acuan: {sample['reference']}")
            prediction = sample["prediction"] or "(kosong)"
            if len(prediction) > 600:
                prediction = prediction[:600] + " ... [dipersingkat untuk tampilan]"
            print(f"Prediksi: {prediction}\n")

train_rows = eligible_rows(train_rows, "train")
print(f"Dev: {len(dev_rows):,} audio digunakan, termasuk audio lebih dari 30 detik.")


## Whisper sebelum fine-tuning

Ukur Whisper Base asli pada dev sebagai pembanding.


In [ ]:
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
base_model.generation_config.forced_decoder_ids = None
baseline_metrics, baseline_examples = evaluate(base_model, dev_rows, "Whisper asli · dev")
show_results("Whisper asli · dev", baseline_metrics, baseline_examples)


## LoRA dan validasi

Latih adapter dengan train dan simpan checkpoint tiap epoch. Pilih checkpoint berdasarkan rata-rata WER dev.


In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
lora = LoraConfig(r=CONFIG["rank"], lora_alpha=CONFIG["alpha"],
                  lora_dropout=CONFIG["dropout"], target_modules=CONFIG["targets"], bias="none")
del base_model
torch.cuda.empty_cache()
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
base_model.generation_config.forced_decoder_ids = None
model = get_peft_model(base_model, lora)
model.print_trainable_parameters()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)
RUN_DIR.mkdir(parents=True, exist_ok=True)
settings = dict(CONFIG, train=fingerprint(train_rows), dev=fingerprint(dev_rows))
latest = RUN_DIR / "latest.pt"
best = RUN_DIR / "best.pt"
if not latest.is_file() and (best.is_file() or any(RUN_DIR.glob("dev_epoch_*.json"))):
    raise FileNotFoundError("latest.pt hilang. Periksa checkpoint sebelum melanjutkan.")
start_epoch, best_score = 1, float("inf")
if latest.is_file():
    saved = torch.load(latest, map_location=DEVICE, weights_only=True)
    if saved["settings"] != settings:
        raise ValueError("Konfigurasi atau data berubah. Gunakan RUN_ID baru.")
    if not best.is_file():
        raise FileNotFoundError("Checkpoint best.pt tidak ditemukan.")
    set_peft_model_state_dict(model, saved["adapter"])
    optimizer.load_state_dict(saved["optimizer"])
    best_score, start_epoch = saved["best_score"], saved["epoch"] + 1
    torch.set_rng_state(saved["torch_rng"].cpu())
    if DEVICE.type == "cuda" and saved["cuda_rng"]:
        torch.cuda.set_rng_state_all([state.cpu() for state in saved["cuda_rng"]])
    print("Melanjutkan dari epoch", start_epoch)

for epoch in range(start_epoch, EPOCHS + 1):
    display(Markdown(f"---\n## Epoch {epoch} / {EPOCHS}\n### Training"))
    model.train()
    started = time.perf_counter()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    shuffled = list(train_rows)
    random.Random(SEED + epoch).shuffle(shuffled)
    optimizer.zero_grad(set_to_none=True)
    losses = []
    batches = prefetch_features(shuffled)
    for step, ((language, path, text, _), (features, attention_mask)) in enumerate(
        progress_batches(batches, "Training", total=len(shuffled)), 1
    ):
        if features.shape[-1] > 3000:
            raise ValueError(f"Audio train di atas 30 detik perlu segmentasi: {path}")
        features = torch.from_numpy(features).unsqueeze(0).to(DEVICE)
        attention_mask = torch.from_numpy(attention_mask).unsqueeze(0).to(DEVICE)
        labels = tokenizers[language](text, return_tensors="pt").input_ids.to(DEVICE)
        if labels[0, 0].item() == model.config.decoder_start_token_id:
            labels = labels[:, 1:]
        if labels.shape[1] > model.config.max_target_positions:
            raise ValueError(f"Transkrip terlalu panjang: {path}")
        loss = model(input_features=features, attention_mask=attention_mask, labels=labels).loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"Loss tidak valid: {path}")
        group_size = min(ACCUMULATION, len(shuffled) - ((step - 1) // ACCUMULATION) * ACCUMULATION)
        (loss / group_size).backward()
        losses.append(loss.item())
        if step % ACCUMULATION == 0 or step == len(shuffled):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
    train_seconds = time.perf_counter() - started
    train_loss = sum(losses) / len(losses)
    print(f"Loss train: {train_loss:.3f} · {train_seconds:.1f} detik · "
          f"{len(shuffled)/train_seconds:.1f} sampel/detik")
    display(Markdown("### Validasi dev"))
    metrics, examples = evaluate(model, dev_rows, "Validasi dev")
    metrics.update(epoch=epoch, train_loss=train_loss,
                   train_seconds=round(train_seconds, 2),
                   peak_vram_mib=round(torch.cuda.max_memory_allocated() / 2**20, 1) if DEVICE.type == "cuda" else None)
    show_results(f"Epoch {epoch} · dev", metrics, examples)
    print(f"Puncak VRAM PyTorch: {metrics['peak_vram_mib']:.0f} MiB")
    improved = metrics["macro_wer"] < best_score
    best_score = min(best_score, metrics["macro_wer"])
    snapshot = {"adapter": get_peft_model_state_dict(model), "optimizer": optimizer.state_dict(),
                "settings": settings, "epoch": epoch, "best_score": best_score,
                "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else []}
    for destination in ([best, latest] if improved else [latest]):
        temporary = destination.with_suffix(".tmp")
        torch.save(snapshot, temporary)
        temporary.replace(destination)
    result_file = RUN_DIR / f"dev_epoch_{epoch:03}.json"
    temporary = result_file.with_suffix(".tmp")
    temporary.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    temporary.replace(result_file)
    print(f"Checkpoint epoch {epoch} tersimpan.")

# Grafik riwayat validasi dev.
import matplotlib.pyplot as plt
history_files = sorted(RUN_DIR.glob("dev_epoch_*.json"))
history = [(int(path.stem.rsplit("_", 1)[1]), json.loads(path.read_text(encoding="utf-8")))
           for path in history_files]
if history:
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    for language, color in (("id_id", "#007C83"), ("en_us", "#6D54AD")):
        for metric, style in (("wer", "-"), ("cer", "--")):
            ax.plot([epoch for epoch, _ in history],
                    [result["per_language"][language][metric] * 100 for _, result in history],
                    linestyle=style, color=color,
                    label=f"{LANGUAGE_NAMES[language]} {metric.upper()}")
    if best.is_file():
        best_epoch = torch.load(best, map_location="cpu", weights_only=True)["epoch"]
        ax.axvline(best_epoch, color="#555555", linestyle=":",
                   label=f"Terbaik: epoch {best_epoch}")
    ax.set_title("WER dan CER dev per epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Error (%)")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

# Bandingkan Whisper asli dan LoRA terbaik pada dev.
if history and "baseline_metrics" in globals():
    best_epoch = torch.load(best, map_location="cpu", weights_only=True)["epoch"]
    best_history = next((result for epoch, result in history if epoch == best_epoch), None)
    if best_history is not None:
        import numpy as np
        labels = ["Indonesia\\nWER", "Indonesia\\nCER", "Inggris\\nWER", "Inggris\\nCER"]
        before = [baseline_metrics["per_language"][language][metric] * 100
                  for language in LANGUAGES for metric in ("wer", "cer")]
        after = [best_history["per_language"][language][metric] * 100
                 for language in LANGUAGES for metric in ("wer", "cer")]
        positions, width = np.arange(len(labels)), 0.34
        fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
        before_bars = ax.bar(positions - width/2, before, width, label="Whisper asli", color="#007C83")
        after_bars = ax.bar(positions + width/2, after, width, label=f"LoRA epoch {best_epoch}", color="#6D54AD")
        ax.bar_label(before_bars, fmt="%.1f%%", padding=3)
        ax.bar_label(after_bars, fmt="%.1f%%", padding=3)
        ax.set_xticks(positions, labels)
        ax.set_title("Whisper asli dan LoRA terbaik pada dev")
        ax.set_ylabel("Error (%)")
        ax.set_ylim(0, max(before + after + [1]) * 1.2)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
        ax.legend()
        plt.show()


## Test akhir

Jalankan setelah checkpoint dipilih dari dev. Catat bahwa split test Model A sudah pernah dibuka.


In [ ]:
if not best.is_file():
    raise FileNotFoundError("Selesaikan training dan validasi terlebih dahulu.")
chosen = torch.load(best, map_location=DEVICE, weights_only=True)
set_peft_model_state_dict(model, chosen["adapter"])
test_metrics, test_examples = evaluate(model, read_split("test"), "Test Model B")
show_results(f"Test · checkpoint epoch {chosen['epoch']}", test_metrics, test_examples)
